In [1]:
import pandas as pd
import json
import os

print("🔄 Chargement des données...")
df = pd.read_csv('co2_data.csv')

# RENOMMER 'Name' en 'country'
df = df.rename(columns={'Name': 'country'})
print(f"✅ {len(df)} lignes chargées")

# ÉTAPE 1: Filtrage temporel
print("📅 Filtrage des années 2000-2023...")
df = df[(df['year'] >= 2000) & (df['year'] <= 2023)]
print(f"   → {len(df)} lignes conservées")

# ÉTAPE 2: Colonnes essentielles
cols = [
    'country', 'iso_code', 'year', 'population', 'gdp',
    'co2', 'co2_per_capita',
    'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2',
    'cumulative_co2', 'consumption_co2',
    'co2_growth_prct', 'trade_co2'
]
df = df[cols]

# ÉTAPE 3: Nettoyage
print("🧹 Nettoyage des données...")
initial_count = len(df)
df = df.dropna(subset=['co2', 'iso_code'])
print(f"   → {initial_count - len(df)} lignes supprimées (valeurs manquantes)")

# ÉTAPE 4: Sélection des pays stratégiques
print("🌍 Sélection des pays pertinents...")

# Top 50 émetteurs
top_50 = df.groupby('country')['co2'].sum().nlargest(50).index.tolist()

# Pays intéressants pour l'analyse
interesting = [
    'France', 'Germany', 'United Kingdom', 'Italy', 'Spain',  # Europe
    'Morocco', 'Algeria', 'Egypt', 'Senegal', 'South Africa',  # Afrique
    'Norway', 'Sweden', 'Denmark', 'Finland',  # Nordiques (décarbonation)
    'Brazil', 'Argentina', 'Chile',  # Amérique du Sud
    'Japan', 'South Korea', 'Singapore',  # Asie développée
    'Vietnam', 'Thailand', 'Indonesia'  # Asie émergente
]

keep = list(set(top_50 + interesting))
df = df[df['country'].isin(keep)]
print(f"   → {len(df['country'].unique())} pays conservés")

# ÉTAPE 5: Compression avec codes pays
print("🗜️ Compression des données...")
countries = sorted(df['country'].unique())
country_codes = {country: idx for idx, country in enumerate(countries)}

# Sauvegarder table de correspondance
pd.DataFrame(
    list(country_codes.items()), 
    columns=['country', 'code']
).to_csv('country_codes.csv', index=False)

# Remplacer noms par codes
df['country_code'] = df['country'].map(country_codes)
df = df.drop('country', axis=1)

# Réorganiser colonnes (country_code en premier)
cols_ordered = ['country_code', 'iso_code', 'year'] + [c for c in df.columns if c not in ['country_code', 'iso_code', 'year']]
df = df[cols_ordered]

# ÉTAPE 6: Arrondir pour réduire la taille
print("🔢 Arrondissement des valeurs numériques...")
numeric_cols = [
    'co2', 'co2_per_capita', 'coal_co2', 'oil_co2', 'gas_co2', 
    'cement_co2', 'cumulative_co2', 'consumption_co2', 
    'co2_growth_prct', 'trade_co2'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].round(2)

df['gdp'] = df['gdp'].round(0)
df['population'] = df['population'].round(0)

# ÉTAPE 7: Générer data.js
print("📝 Génération de data.js...")
data_json = df.to_dict(orient='records')

# Créer le dossier si nécessaire
os.makedirs('../site/js', exist_ok=True)

with open('../site/js/data.js', 'w', encoding='utf-8') as f:
    f.write('/**\n')
    f.write(' * Données CO2 - Challenge Open Data\n')
    f.write(' * Source: Kaggle CO2 Emissions Dataset\n')
    f.write(f' * Période: 2000-2023\n')
    f.write(f' * Pays: {len(countries)}\n')
    f.write(f' * Lignes: {len(df)}\n')
    f.write(' */\n\n')
    
    # Données principales
    f.write('const DATA = ')
    f.write(json.dumps(data_json, ensure_ascii=False))
    f.write(';\n\n')
    
    # Table de correspondance pays
    f.write('const COUNTRY_CODES = ')
    f.write(json.dumps(country_codes, ensure_ascii=False, indent=2))
    f.write(';\n\n')
    
    # Inverser pour accès rapide code → nom
    f.write('// Décoder les codes pays (code → nom)\n')
    f.write('const COUNTRIES = {};\n')
    f.write('Object.entries(COUNTRY_CODES).forEach(([name, code]) => {\n')
    f.write('    COUNTRIES[code] = name;\n')
    f.write('});\n\n')
    
    # Liste des ISO codes pour le GeoJSON
    f.write('// Liste des ISO codes présents\n')
    iso_codes = sorted(df['iso_code'].unique())
    f.write(f'const ISO_CODES = {json.dumps(iso_codes)};\n\n')
    
    # NOUVEAU: Ajouter ranges pour les scales
    f.write('// Ranges pour les échelles\n')
    f.write('const DATA_RANGES = {\n')
    f.write(f'    years: [{df["year"].min()}, {df["year"].max()}],\n')
    f.write(f'    co2: [{df["co2"].min():.2f}, {df["co2"].max():.2f}],\n')
    f.write(f'    co2_per_capita: [{df["co2_per_capita"].min():.2f}, {df["co2_per_capita"].max():.2f}],\n')
    f.write(f'    gdp: [{df["gdp"].min():.0f}, {df["gdp"].max():.0f}],\n')
    f.write(f'    population: [{df["population"].min():.0f}, {df["population"].max():.0f}]\n')
    f.write('};\n')

# ÉTAPE 8: Statistiques finales
file_size = os.path.getsize('../site/js/data.js') / (1024 * 1024)
print(f"\n{'='*50}")
print(f"🎉 GÉNÉRATION TERMINÉE !")
print(f"{'='*50}")
print(f"📊 Statistiques:")
print(f"   • Lignes de données: {len(df):,}")
print(f"   • Pays: {len(countries)}")
print(f"   • Période: 2000-2023 ({2023-2000+1} années)")
print(f"   • Colonnes: {len(df.columns)}")
print(f"   • Taille fichier: {file_size:.2f} MB")
print(f"\n📁 Fichier créé: ../site/js/data.js")
print(f"📋 Table codes: country_codes.csv")
print(f"{'='*50}")

# ÉTAPE 9: Quelques stats intéressantes
print(f"\n📈 Top 5 émetteurs en 2023:")
df_2023 = df[df['year'] == 2023].sort_values('co2', ascending=False).head(5)
for idx, row in df_2023.iterrows():
    country_name = countries[row['country_code']]
    print(f"   {country_name}: {row['co2']:.2f} Mt CO2")

print(f"\n✅ Prêt pour le développement web !")

🔄 Chargement des données...
✅ 43746 lignes chargées
📅 Filtrage des années 2000-2023...
   → 6072 lignes conservées
🧹 Nettoyage des données...
   → 912 lignes supprimées (valeurs manquantes)
🌍 Sélection des pays pertinents...
   → 57 pays conservés
🗜️ Compression des données...
🔢 Arrondissement des valeurs numériques...
📝 Génération de data.js...

🎉 GÉNÉRATION TERMINÉE !
📊 Statistiques:
   • Lignes de données: 1,368
   • Pays: 57
   • Période: 2000-2023 (24 années)
   • Colonnes: 15
   • Taille fichier: 0.42 MB

📁 Fichier créé: ../site/js/data.js
📋 Table codes: country_codes.csv

📈 Top 5 émetteurs en 2023:
   China: 11902.50 Mt CO2
   United States: 4911.39 Mt CO2
   India: 3062.32 Mt CO2
   Russia: 1815.92 Mt CO2
   Japan: 988.78 Mt CO2

✅ Prêt pour le développement web !
